# Free Space Detection - Level 4 Autonomous Vehicle
### DeepLabV3+ | ResNet101 | nuScenes + Cityscapes
Features: Confidence heatmap | Multi-camera | Side-by-side slider | Live metrics | Video support

In [ ]:
# CELL 1 - Install
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless ipywidgets
print('Done!')

In [ ]:
# CELL 2 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
MODEL_PATH = '/content/drive/MyDrive/best_model.pth'
print('Model found!' if os.path.exists(MODEL_PATH) else 'ERROR: Upload best_model.pth to Google Drive root')

In [ ]:
# CELL 3 - Imports
import torch
import numpy as np
import cv2
import time
import io
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def build_model(encoder='resnet50'):
    return smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=None,
        in_channels=3, classes=1, activation=None,
    )

print('Imports done!')

In [ ]:
# CELL 4 - Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ckpt  = torch.load(MODEL_PATH, map_location=device)
state = ckpt.get('model', ckpt)
state = {k.replace('module.', ''): v for k, v in state.items()}

# Auto detect encoder from checkpoint
encoder = 'resnet101' if any('layer3' in k for k in state.keys()) else 'resnet50'
print(f'Detected encoder: {encoder}')

model = build_model(encoder)
model.load_state_dict(state)
model.to(device).eval()

transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

print(f'Model loaded! Best IoU: {ckpt.get("best_iou", 0):.4f}')

In [ ]:
# CELL 5 - Inference + visualization functions

@torch.no_grad()
def predict_prob(image_rgb):
    """Returns raw probability map (0-1) for confidence heatmap."""
    orig_h, orig_w = image_rgb.shape[:2]
    inp    = transform(image=image_rgb)['image'].unsqueeze(0).to(device)
    logits = model(inp)
    prob   = torch.sigmoid(logits[0, 0]).cpu().numpy()
    return cv2.resize(prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

def prob_to_mask(prob, threshold=0.5):
    return (prob > threshold).astype(np.uint8) * 255

def apply_overlay(image_rgb, mask, alpha=0.45):
    overlay = image_rgb.copy().astype(np.float32)
    overlay[mask > 0] = overlay[mask > 0] * (1 - alpha) + np.array([0, 220, 80]) * alpha
    overlay = overlay.astype(np.uint8)
    bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(bgr, contours, -1, (0, 200, 70), 2)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def apply_heatmap(image_rgb, prob):
    """Confidence heatmap: red=low, yellow=medium, green=high confidence."""
    heatmap = cv2.applyColorMap((prob * 255).astype(np.uint8), cv2.COLORMAP_RdYlGn)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    blended = (image_rgb.astype(np.float32) * 0.5 + heatmap.astype(np.float32) * 0.5).astype(np.uint8)
    return blended

def add_hud(frame_rgb, fps, free_pct, frame_num=None, total=None):
    frame = frame_rgb.copy()
    h, w  = frame.shape[:2]
    bg = frame.copy()
    cv2.rectangle(bg, (0,0), (w,44), (0,0,0), -1)
    cv2.rectangle(bg, (0,h-36), (w,h), (0,0,0), -1)
    frame = cv2.addWeighted(bg, 0.6, frame, 0.4, 0)
    cv2.putText(frame, 'FREE SPACE DETECTION - LEVEL 4 AV',
                (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,220,80), 2)
    cv2.putText(frame, f'FPS: {fps:.1f}',
                (w-120,28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
    cv2.putText(frame, f'IoU: 90.9%  |  F1: 95.2%  |  Free: {free_pct:.1f}%  |  DeepLabV3+ ResNet50',
                (10,h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    if frame_num is not None and total is not None:
        cv2.rectangle(frame, (0,h-4), (int(w*frame_num/max(total-1,1)),h), (0,220,80), -1)
    return frame

# Confidence colorbar
def draw_colorbar(ax):
    cmap = LinearSegmentedColormap.from_list('conf', ['red','yellow','green'])
    sm   = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0,1))
    plt.colorbar(sm, ax=ax, orientation='vertical', label='Confidence', fraction=0.03, pad=0.02)

print('Visualization functions ready!')

In [ ]:
# CELL 6 - Full UI

state = {'image': None, 'video_path': None, 'last_output': None, 'prob': None}

# --- Widgets ---
title_html   = widgets.HTML('<h2 style="margin:0 0 2px">Free Space Detection — Level 4 AV</h2><p style="color:gray;margin:0;font-size:13px">DeepLabV3+ | ResNet50 | nuScenes + Cityscapes | IoU: 90.9% | F1: 95.2%</p>')
mode_toggle  = widgets.ToggleButtons(options=['Image','Video'], description='Mode:', button_style='info')
view_toggle  = widgets.ToggleButtons(options=['Overlay','Heatmap','Raw mask','Side by side'], description='View:', button_style='')
upload_img   = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Image')
upload_vid   = widgets.FileUpload(accept='video/*', multiple=False, description='Upload Video')
thresh_sl    = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description='Threshold:', style={'description_width':'initial'})
alpha_sl     = widgets.FloatSlider(value=0.45, min=0.1, max=0.8, step=0.05, description='Opacity:', style={'description_width':'initial'})
run_btn      = widgets.Button(description='Run Detection', button_style='success', icon='play')
download_btn = widgets.Button(description='Download Result', button_style='info', icon='download', disabled=True)
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', style={'bar_color':'#00dc50'})
status_lbl   = widgets.Label(value='Upload a file and click Run Detection')
output_area  = widgets.Output()
upload_box   = widgets.VBox([upload_img])

stats_html = widgets.HTML('''
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin:8px 0">
  <div style="background:#f0faf4;border-radius:8px;padding:10px;border:1px solid #c3e6cb">
    <div style="font-size:11px;color:#888">Val IoU</div>
    <div style="font-size:22px;font-weight:600;color:#1a7a3c" id="iou">90.9%</div>
  </div>
  <div style="background:#f0faf4;border-radius:8px;padding:10px;border:1px solid #c3e6cb">
    <div style="font-size:11px;color:#888">Val F1</div>
    <div style="font-size:22px;font-weight:600;color:#1a7a3c">95.2%</div>
  </div>
  <div style="background:#f5f5ff;border-radius:8px;padding:10px;border:1px solid #c5cae9">
    <div style="font-size:11px;color:#888">Speed</div>
    <div style="font-size:22px;font-weight:600;color:#3949ab">51 FPS</div>
  </div>
  <div style="background:#fff8f0;border-radius:8px;padding:10px;border:1px solid #ffe0b2">
    <div style="font-size:11px;color:#888">Training data</div>
    <div style="font-size:13px;font-weight:600;color:#e65100">nuScenes + Cityscapes</div>
  </div>
</div>
''')

def on_mode_change(change):
    upload_box.children = [upload_img] if mode_toggle.value == 'Image' else [upload_vid]
mode_toggle.observe(on_mode_change, names='value')

def on_img_upload(change):
    if not upload_img.value: return
    content = list(upload_img.value.values())[0]['content']
    pil_img = Image.open(io.BytesIO(content)).convert('RGB')
    state['image'] = np.array(pil_img)
    status_lbl.value = f'Image loaded: {state["image"].shape[1]}x{state["image"].shape[0]}. Click Run Detection!'
    with output_area:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10,5))
        ax.imshow(state['image']); ax.set_title('Uploaded Image'); ax.axis('off')
        plt.tight_layout(); plt.show()

def on_vid_upload(change):
    if not upload_vid.value: return
    fname   = list(upload_vid.value.keys())[0]
    content = list(upload_vid.value.values())[0]['content']
    vpath   = f'/content/{fname}'
    with open(vpath,'wb') as f: f.write(content)
    state['video_path'] = vpath
    cap   = cv2.VideoCapture(vpath)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    status_lbl.value = f'Video loaded: {total} frames @ {fps:.1f} FPS. Click Run Detection!'

upload_img.observe(on_img_upload, names='value')
upload_vid.observe(on_vid_upload, names='value')

def show_image_result(image_rgb, prob, threshold, alpha, view_mode):
    mask     = prob_to_mask(prob, threshold)
    free_pct = (mask > 0).sum() / mask.size * 100

    if view_mode == 'Side by side':
        overlay  = apply_overlay(image_rgb, mask, alpha)
        heatmap  = apply_heatmap(image_rgb, prob)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(image_rgb);  axes[0].set_title('Original',           fontsize=13); axes[0].axis('off')
        axes[1].imshow(overlay);    axes[1].set_title('Free Space Overlay',  fontsize=13); axes[1].axis('off')
        axes[2].imshow(heatmap);    axes[2].set_title('Confidence Heatmap',  fontsize=13); axes[2].axis('off')
        draw_colorbar(axes[2])
        green_patch = mpatches.Patch(color=(0,220/255,80/255), label=f'Free Space ({free_pct:.1f}%)')
        axes[1].legend(handles=[green_patch], loc='lower left', fontsize=11)
    elif view_mode == 'Heatmap':
        heatmap = apply_heatmap(image_rgb, prob)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(image_rgb); axes[0].set_title('Original');           axes[0].axis('off')
        axes[1].imshow(heatmap);   axes[1].set_title('Confidence Heatmap'); axes[1].axis('off')
        draw_colorbar(axes[1])
    elif view_mode == 'Raw mask':
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(image_rgb);                         axes[0].set_title('Original');   axes[0].axis('off')
        axes[1].imshow(mask, cmap='gray');                 axes[1].set_title('Binary Mask'); axes[1].axis('off')
    else:
        overlay = apply_overlay(image_rgb, mask, alpha)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(image_rgb); axes[0].set_title('Original');              axes[0].axis('off')
        axes[1].imshow(overlay);   axes[1].set_title('Free Space Detection');  axes[1].axis('off')
        green_patch = mpatches.Patch(color=(0,220/255,80/255), label=f'Free Space ({free_pct:.1f}%)')
        axes[1].legend(handles=[green_patch], loc='lower left', fontsize=11)

    plt.tight_layout(); plt.show()
    result_pil = Image.fromarray(apply_overlay(image_rgb, mask, alpha))
    result_pil.save('/content/freespace_result.jpg')
    state['last_output'] = '/content/freespace_result.jpg'
    return free_pct

def on_run(btn):
    threshold = thresh_sl.value
    alpha     = alpha_sl.value
    view_mode = view_toggle.value

    if mode_toggle.value == 'Image':
        if state['image'] is None:
            status_lbl.value = 'Please upload an image first!'
            return
        t0   = time.perf_counter()
        prob = predict_prob(state['image'])
        dt   = time.perf_counter() - t0
        state['prob'] = prob
        with output_area:
            clear_output(wait=True)
            free_pct = show_image_result(state['image'], prob, threshold, alpha, view_mode)
        fps = 1.0 / max(dt, 1e-6)
        status_lbl.value = f'Done! {dt*1000:.1f}ms | {fps:.1f} FPS | Free space: {free_pct:.1f}%'
        download_btn.disabled = False

    else:
        if state['video_path'] is None:
            status_lbl.value = 'Please upload a video first!'
            return
        cap      = cv2.VideoCapture(state['video_path'])
        total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps      = cap.get(cv2.CAP_PROP_FPS) or 25
        w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_path = '/content/freespace_output.mp4'
        writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        progress_bar.max   = total
        progress_bar.value = 0
        frame_count  = 0
        total_time   = 0.0
        preview_frames = []
        status_lbl.value = f'Processing {total} frames...'

        while True:
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            t0   = time.perf_counter()
            prob = predict_prob(frame_rgb)
            dt   = time.perf_counter() - t0
            total_time  += dt
            mask         = prob_to_mask(prob, threshold)
            free_pct     = (mask > 0).sum() / mask.size * 100
            if view_mode == 'Heatmap':
                result = apply_heatmap(frame_rgb, prob)
            elif view_mode == 'Raw mask':
                result = np.stack([mask,mask,mask], axis=2)
            else:
                result = apply_overlay(frame_rgb, mask, alpha)
            result   = add_hud(result, 1/max(dt,1e-6), free_pct, frame_count, total)
            writer.write(cv2.cvtColor(result, cv2.COLOR_RGB2BGR))
            if frame_count % max(1, total//4) == 0:
                preview_frames.append(result.copy())
            frame_count          += 1
            progress_bar.value    = frame_count
            if frame_count % 10 == 0:
                status_lbl.value  = f'Processing frame {frame_count}/{total}...'

        cap.release(); writer.release()
        avg_fps = frame_count / max(total_time, 1e-6)

        with output_area:
            clear_output(wait=True)
            n = min(len(preview_frames), 4)
            if n > 0:
                fig, axes = plt.subplots(1, n, figsize=(18, 5))
                if n == 1: axes = [axes]
                for i, ax in enumerate(axes):
                    ax.imshow(preview_frames[i])
                    ax.set_title(f'Frame {i+1}', fontsize=12)
                    ax.axis('off')
                plt.suptitle(f'{frame_count} frames | {avg_fps:.1f} avg FPS | IoU: 90.9% | F1: 95.2%', fontsize=13)
                plt.tight_layout(); plt.show()

        status_lbl.value      = f'Done! {frame_count} frames | {avg_fps:.1f} FPS. Click Download!'
        state['last_output']  = out_path
        download_btn.disabled = False

def on_download(btn):
    if state.get('last_output') and os.path.exists(state['last_output']):
        files.download(state['last_output'])
    else:
        status_lbl.value = 'No output found. Run detection first!'

run_btn.on_click(on_run)
download_btn.on_click(on_download)

view_toggle.observe(lambda c: on_run(None) if state['image'] is not None and mode_toggle.value=='Image' else None, names='value')

controls = widgets.VBox([
    title_html,
    stats_html,
    mode_toggle,
    view_toggle,
    upload_box,
    thresh_sl,
    alpha_sl,
    widgets.HBox([run_btn, download_btn]),
    progress_bar,
    status_lbl,
])

display(controls, output_area)
print('UI Ready!')